In [ ]:
%pip install openai pdfplumber python-docx pillow
from openai import OpenAI

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True, dotenv_path="../.env.local")
api_key=os.getenv("OPENAI_API_KEY")
print(api_key[-5:])
client=OpenAI(api_key=api_key)

In [ ]:
import pdfplumber, docx, base64

client = OpenAI()

## Extract the information from PDF

### Reads text with pdfplumber and sends to GPT-5-nano for summarization.

In [ ]:
pdf_path = "data/California_Employment_Offer_Letter.pdf"  

def extract_text_from_pdf(path):
    complete_text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                complete_text += text + "\n"
    return complete_text.strip()

pdf_text = extract_text_from_pdf(pdf_path)
pdf_text

In [12]:
#Open AI API call to summarize and extract key details from the PDF text
prompt = f"Extract and summarize key details from this PDF:\n\n{pdf_text}"

response_gpt = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "You summarize and extract data from PDF text."},
        {"role": "user", "content": prompt}
    ]
)

print("Extracted Info from PDF:\n")
print(response_gpt.choices[0].message.content)
#print(response['message']['content'])

Extracted Info from PDF:

Here are the key details and a concise summary extracted from the employment offer letter.

Key parties and document
- Company: Acme Corporation, 1234 Market Street, San Francisco, CA 94105
- Candidate: Jane Doe, 567 Oak Avenue, San Jose, CA 95112
- Date of offer: April 1, 2023
- Title of offer: Employment offer letter (California edition)

Position, reporting, location, and travel
- Position: Senior Software Engineer
- Duties: Designing, coding, testing, maintaining enterprise applications; cross-functional collaboration; mentoring; ensuring security and data privacy standards
- Reports to: John Smith, Chief Technology Officer
- Status: Full-time, exempt
- Primary work location: San Francisco, CA
- Travel: Up to 10% of time

Start date and offer expiration
- Start date: April 24, 2023 (contingent on conditions in Section 6)
- Offer expiration / acceptance deadline: April 15, 2023 (signed acceptance required by then)

 compensation
- Base salary: $150,000 per 

In [13]:

import ollama
#Open AI API call to summarize and extract key details from the PDF text
prompt = f"Extract and summarize key details from this PDF:\n\n{pdf_text}"

response_ollama = ollama.chat(
    model="llama3",
    messages=[
        {"role": "system", "content": "You summarize and extract data from PDF text."},
        {"role": "user", "content": prompt}
    ]
)

print("Extracted Info from PDF:\n")
#print(response.choices[0].message.content)
print(response_ollama['message']['content'])

Extracted Info from PDF:

Here are the key details extracted from the PDF:

**Job Offer Details**

* Job Title: Senior Software Engineer
* Reporting Manager: John Smith, Chief Technology Officer
* Primary Work Location: San Francisco, CA
* Travel Requirement: Up to 10% of time
* Start Date: April 24, 2023 (contingent upon satisfaction of conditions listed in Section 6)
* Offer Expiration Date: April 15, 2023

**Compensation**

* Base Salary: $150,000 per year (bi-weekly installments)
* Bonus: Targeted at 10% of base salary
* Equity Grant: 10,000 Restricted Stock Units (RSUs), vesting over 4 years with a 1-year cliff (subject to Board approval)

**Benefits and Paid Time Off**

* Health, dental, vision insurance
* 401(k) with 4% match
* 15 vacation days per year
* 10 paid holidays per year
* Sick leave in accordance with California Healthy Workplaces, Healthy Families Act

**Confidentiality, IP, and Conflict of Interest**

* Required to sign Confidential Information and Invention Assignm

In [17]:
#Open AI API call to summarize and extract key details from the PDF text
prompt = f"Extract and summarize key details from this PDF:\n\n{pdf_text}"

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": ''' You are a helpful assistant, compare 2 response outputs and assign a score
          between 0 to 100 based on effectiveness and provide reasoning'''},
        {"role": "user", "content": f"Compare below responses: \n\n response_gpt:{response_gpt} \n\n response_ollama:{response_ollama}"}
    ]
)

print("Extracted Info from PDF:\n")
print(response.choices[0].message.content)
#print(response['message']['content'])

Extracted Info from PDF:

Here is a comparison and scoring for the two responses, with reasoning and concrete notes.

Overall scores
- response_gpt: 82/100
- response_ollama: 42/100

Reasoning and detailed notes

response_gpt
Strengths
- Clear purpose: Immediately addresses the user’s intent to compare responses and offers a practical, repeatable rubric for evaluation.
- Structured framework: Provides comprehensive criteria (accuracy, completeness, relevance, clarity, etc.) and a detailed scoring rubric (0–5 per criterion) with weighted scoring options. This makes evaluation transparent and reproducible.
- Guidance for execution: Includes concrete steps for scoring, what to extract, and output format suggestions (side-by-side comparison, notes, recommendations).
- Self-awareness: Acknowledges the need for the actual texts to perform the comparison and invites the user to paste them, which is honest and user-friendly.

Weaknesses / areas for improvement
- Limited immediate usefulness: B

## Extract Information from a DOCX (Word) File
### Reads paragraphs and asks GPT-5-nano for key points.

In [18]:
docx_path = "data/Project_List.docx" 

def extract_text_from_docx(path):
    document = docx.Document(path)
    return "\n".join([p.text for p in document.paragraphs])

docx_text = extract_text_from_docx(docx_path)

print (f"Extracted DOCX Text (first 500 chars):\n{docx_text[:500]}\n")
prompt = f"Extract key points from this DOCX content:\n\n{docx_text[:8000]}"

Extracted DOCX Text (first 500 chars):

Project 1

Project: Real Estate Platform - eREP
Company: Sansa Technology LLC, Milpitas, CA, USA
Duration: From March 2017 – Current
Role: Java EE Developer
Description:
eREP provides leading real estate and rental marketplace platform. eREP serves the full lifecycle of owning and living in a home: buying, selling, renting, financing, remodeling and more. 
Responsibilities:
• Analyzed requirements and designed class diagrams, sequence diagrams using UML and
prepared high level technical documen



In [19]:
#Open AI API call to summarize and extract key details from the DOCX text
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": '''You summarize and extract structured data from Word documents. Also, extract 
         a list of skills used in that project. Reurn as JSON with the following format:
            {
            "projects": [
                {
                "name": "Project Name",
                "description": "Brief description of the project",
                "skills": ["Skill1", "Skill2", "Skill3"]
                },
                ...
            ]
            }
        '''},
        {"role": "user", "content": prompt}
    ]
)

print("Extracted Info from DOCX:\n")
print(response.choices[0].message.content)

Extracted Info from DOCX:

{
  "projects": [
    {
      "name": "Real Estate Platform - eREP",
      "description": "Leading real estate and rental marketplace platform covering the full lifecycle of owning and living in a home (buying, selling, renting, financing, remodeling, etc.).",
      "skills": [
        "Java",
        "Java EE",
        "UML",
        "Class Diagrams",
        "Sequence Diagrams",
        "Spring MVC",
        "Dependency Injection",
        "Hibernate",
        "MySQL",
        "SQL",
        "REST Web Services",
        "JAX-RS",
        "JAXB",
        "JSP",
        "HTML",
        "CSS",
        "JavaScript",
        "AngularJS",
        "Jenkins",
        "Tomcat",
        "JUnit",
        "XML",
        "J2EE Design Patterns"
      ]
    },
    {
      "name": "Online Advertising Platform - OAP",
      "description": "Classifieds listings platform (similar to Craigslist) for posting free or paid classifieds, with categories such as Housing, Lessons, an